# Simplified Re‑ID Pipeline (Online Tracking + Offline Clustering)

This notebook tests an **alternative approach**:
- Run YOLO + ByteTrack to get local tracks.
- Save crops per **local track ID** (no online ReID).
- **Offline**: load all tracks, filter bad crops, compute per‑track prototypes, then cluster tracks into final identities using appearance similarity and frame‑overlap constraints.

All thresholds are read from `config.py` and can be overridden below.

In [1]:
# ========== EDIT THESE ==========
VIDEO_PATH = "../assets/store_cam1.mp4"
OUTPUT_DIR = "./outputs/full_pipeline_cam1"

# Override config thresholds (None = use config.py value)
# For display / save gates (online):
SAVE_MIN_WIDTH        = None
SAVE_MIN_HEIGHT       = None
SAVE_MIN_AREA_RATIO   = None   # e.g., 0.0010
SAVE_MIN_SHARPNESS    = None   # e.g., 8.0

# For offline refinement (used to filter crops before clustering):
REFINE_MIN_AREA_RATIO = None   # e.g., 0.0012
REFINE_MIN_SHARPNESS  = None   # e.g., 12.0
REFINE_MAX_ASPECT     = None   # e.g., 5.0 (0 = disabled)

# Clustering parameters (used offline):
MERGE_REID_THRESHOLD  = None   # e.g., 0.85
MERGE_COLOR_THRESHOLD = None   # e.g., 0.40
MERGE_COLOR_WEIGHT    = None   # e.g., 0.20
MAX_SAME_FRAME_OVERLAP = None  # e.g., 3
COHERENCE_MIN         = None   # e.g., 0.80
MIN_FOLDER_IMAGES_FINAL = None # e.g., 3
LARGE_FOLDER_SIZE     = None   # e.g., 100
LARGE_MERGE_REID_THRESHOLD = None # e.g., 0.85
LARGE_MERGE_COLOR_THRESHOLD = None # e.g., 0.35
# ==================================

In [2]:
import sys, time, os
from pathlib import Path
import numpy as np
import cv2
import torch
from collections import defaultdict
from tqdm.auto import tqdm

# Add project root
project_root = Path(os.getcwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import config
from modules.detector_tracker import YOLOTracker

# Resolve thresholds
def override(name, default):
    return default if eval(name.upper()) is None else eval(name.upper())

SAVE_MIN_WIDTH        = override('SAVE_MIN_WIDTH', config.MIN_WIDTH)
SAVE_MIN_HEIGHT       = override('SAVE_MIN_HEIGHT', config.MIN_HEIGHT)
SAVE_MIN_AREA_RATIO   = override('SAVE_MIN_AREA_RATIO', config.MIN_AREA_RATIO)
SAVE_MIN_SHARPNESS    = override('SAVE_MIN_SHARPNESS', config.MIN_SHARPNESS)
REFINE_MIN_AREA_RATIO = override('REFINE_MIN_AREA_RATIO', config.REFINE_MIN_REID_AREA_RATIO)
REFINE_MIN_SHARPNESS  = override('REFINE_MIN_SHARPNESS', config.REFINE_MIN_REID_SHARPNESS)
REFINE_MAX_ASPECT     = override('REFINE_MAX_ASPECT', config.REFINE_MAX_ASPECT_RATIO)
MERGE_REID_THRESHOLD  = override('MERGE_REID_THRESHOLD', config.MERGE_REID_THRESHOLD)
MERGE_COLOR_THRESHOLD = override('MERGE_COLOR_THRESHOLD', config.MERGE_COLOR_THRESHOLD)
MERGE_COLOR_WEIGHT    = override('MERGE_COLOR_WEIGHT', config.MERGE_COLOR_WEIGHT)
MAX_SAME_FRAME_OVERLAP = override('MAX_SAME_FRAME_OVERLAP', config.MAX_SAME_FRAME_OVERLAP)
COHERENCE_MIN         = override('COHERENCE_MIN', config.COHERENCE_MIN)
MIN_FOLDER_IMAGES_FINAL = override('MIN_FOLDER_IMAGES_FINAL', config.MIN_FOLDER_IMAGES_FINAL)
LARGE_FOLDER_SIZE     = override('LARGE_FOLDER_SIZE', config.LARGE_FOLDER_SIZE)
LARGE_MERGE_REID_THRESHOLD = override('LARGE_MERGE_REID_THRESHOLD', config.LARGE_MERGE_REID_THRESHOLD)
LARGE_MERGE_COLOR_THRESHOLD = override('LARGE_MERGE_COLOR_THRESHOLD', config.LARGE_MERGE_COLOR_THRESHOLD)

print("Thresholds loaded.")

Thresholds loaded.


In [3]:
# ---- Helper functions for ReID & colour (will be used offline) ----
import onnxruntime as ort

def load_osnet(model_path):
    sess = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
    return sess, sess.get_inputs()[0].name

def preprocess_crop(crop, size=(256, 128)):
    h, w = size
    resized = cv2.resize(crop, (w, h))
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    rgb = np.transpose(rgb, (2, 0, 1))
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)
    return (rgb - mean) / std

def extract_embedding(crop, sess, iname, size=(256, 128)):
    tensor = preprocess_crop(crop, size)[np.newaxis, ...]
    emb = sess.run(None, {iname: tensor})[0][0]
    return emb / (np.linalg.norm(emb) + 1e-12)

def extract_color(crop):
    resized = cv2.resize(crop, (64, 128))
    lab = cv2.cvtColor(resized, cv2.COLOR_BGR2LAB)
    split = lab.shape[0] // 2
    upper = lab[:split, :, :]
    lower = lab[split:, :, :]
    hist_u = cv2.calcHist([upper], [0,1,2], None, [8,8,8],
                          [0,256,0,256,0,256]).flatten().astype(np.float32)
    hist_l = cv2.calcHist([lower], [0,1,2], None, [8,8,8],
                          [0,256,0,256,0,256]).flatten().astype(np.float32)
    hist_u /= (hist_u.sum() + 1e-12)
    hist_l /= (hist_l.sum() + 1e-12)
    return hist_u, hist_l

def color_similarity(u1, l1, u2, l2):
    return 0.4 * np.minimum(u1, u2).sum() + 0.6 * np.minimum(l1, l2).sum()

def estimate_sharpness(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_32F).var()

print("Helper functions ready.")

Helper functions ready.


## Step 1 – Online Tracking & Crop Saving (per local ID)

In [4]:
# Prepare output directories
out_base = Path(OUTPUT_DIR)
tracks_dir = out_base / "tracks"
tracks_dir.mkdir(parents=True, exist_ok=True)

# Open video
cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Video: {VIDEO_PATH} | {total_frames} frames @ {fps:.2f} FPS, {w}x{h}")

# Initialise YOLO tracker
tracker = YOLOTracker()

frame_idx = 0
saved_total = 0
saved_per_track = defaultdict(int)
filtered_save_count = 0

pbar = tqdm(total=total_frames, desc="Tracking & saving")
start_time = time.perf_counter()

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1
    dets = tracker.track(frame, persist=True)
    for det in dets:
        x1, y1, x2, y2 = det['bbox']
        bw, bh = x2 - x1, y2 - y1
        lid = det['local_id']
        crop = det['crop']
        area_ratio = (bw * bh) / (w * h)
        # Compute sharpness only if size gates pass (short-circuit)
        sharp = None
        save_ok = (bw >= SAVE_MIN_WIDTH and bh >= SAVE_MIN_HEIGHT and
                   area_ratio >= SAVE_MIN_AREA_RATIO)
        if save_ok:
            sharp = estimate_sharpness(crop)
            save_ok = sharp >= SAVE_MIN_SHARPNESS
        if not save_ok:
            filtered_save_count += 1
            continue
        # Save crop
        track_dir = tracks_dir / f"track_{lid:03d}"
        track_dir.mkdir(parents=True, exist_ok=True)
        ts = frame_idx / fps
        fname = f"frame_{frame_idx:06d}_t{ts:.3f}_l{lid}.jpg"
        cv2.imwrite(str(track_dir / fname), crop,
                    [cv2.IMWRITE_JPEG_QUALITY, config.JPEG_QUALITY])
        saved_total += 1
        saved_per_track[lid] += 1
    pbar.update(1)

pbar.close()
cap.release()
track_time = time.perf_counter() - start_time
print(f"\nTracking finished in {track_time:.2f}s ({track_time/60:.2f} min)")
print(f"Tracks detected: {len(saved_per_track)}")
print(f"Total crops saved: {saved_total}")
print(f"Crops filtered out by save gates: {filtered_save_count}")

# Show top 10 track sizes
top_tracks = sorted(saved_per_track.items(), key=lambda x: x[1], reverse=True)[:10]
print("Top 10 local tracks (by crops saved):")
for lid, cnt in top_tracks:
    print(f"  track_{lid:03d}: {cnt} crops")

Video: ../assets/store_cam1.mp4 | 396 frames @ 25.00 FPS, 1920x1080
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
); using CPU instead.


Tracking & saving:   0%|          | 0/396 [00:00<?, ?it/s]


Tracking finished in 129.57s (2.16 min)
Tracks detected: 24
Total crops saved: 1588
Crops filtered out by save gates: 71
Top 10 local tracks (by crops saved):
  track_001: 316 crops
  track_003: 313 crops
  track_005: 203 crops
  track_002: 154 crops
  track_027: 140 crops
  track_009: 89 crops
  track_004: 74 crops
  track_006: 52 crops
  track_018: 50 crops
  track_029: 38 crops


## Step 2 – Offline Track Clustering
Now we load all track folders, filter crops again (refine gates), compute track prototypes, and then greedily merge tracks into final identities.

In [5]:
# Load ONNX
reid_model_path = Path(config.OSNET_MODEL)
sess, iname = load_osnet(reid_model_path)

# Prepare to collect tracks
track_folders = sorted([d for d in tracks_dir.iterdir() if d.is_dir() and d.name.startswith("track_")],
                       key=lambda d: int(d.name.split("_")[1]))
print(f"Found {len(track_folders)} track folders.")

Found 24 track folders.


In [6]:
# ---- Filter crops and build track prototypes ----
class TrackProto:
    def __init__(self, track_id):
        self.track_id = track_id
        self.image_paths = []
        self.embeddings = []
        self.upper_hist = []
        self.lower_hist = []
        self.frames = []
        self.proto_emb = None
        self.proto_up = None
        self.proto_lo = None
        self.coherence = None

    def add_crop(self, path, emb, up, lo, frame):
        self.image_paths.append(path)
        self.embeddings.append(emb)
        self.upper_hist.append(up)
        self.lower_hist.append(lo)
        self.frames.append(frame)

    def size(self):
        return len(self.image_paths)

    def build_prototype(self):
        if not self.embeddings:
            return
        embs = np.vstack(self.embeddings)
        self.proto_emb = embs.mean(axis=0)
        self.proto_emb /= (np.linalg.norm(self.proto_emb) + 1e-12)
        self.proto_up = np.mean(self.upper_hist, axis=0)
        self.proto_lo = np.mean(self.lower_hist, axis=0)
        s_up, s_lo = self.proto_up.sum(), self.proto_lo.sum()
        if s_up > 0: self.proto_up /= s_up
        if s_lo > 0: self.proto_lo /= s_lo
        self.coherence = np.dot(embs, self.proto_emb).mean()

    def frame_set(self):
        return set(self.frames)

    def similarity_to(self, other):
        reid = float(np.dot(self.proto_emb, other.proto_emb))
        col = color_similarity(self.proto_up, self.proto_lo, other.proto_up, other.proto_lo)
        combined = reid + MERGE_COLOR_WEIGHT * col
        return reid, col, combined

# Process each track folder
tracks = []
filtered_offline = 0

for folder in tqdm(track_folders, desc="Loading tracks"):
    lid = int(folder.name.split("_")[1])
    images = sorted([p for p in folder.iterdir() if p.suffix.lower() in {".jpg",".jpeg",".png"}])
    track = TrackProto(lid)
    for img_path in images:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h_crop, w_crop = img.shape[:2]
        area_ratio = (w_crop * h_crop) / (w * h)
        sharp = estimate_sharpness(img)
        aspect = max(w_crop, h_crop) / max(1, min(w_crop, h_crop))
        # Offline refine gates
        if (area_ratio < REFINE_MIN_AREA_RATIO or sharp < REFINE_MIN_SHARPNESS or
            (REFINE_MAX_ASPECT > 0 and aspect > REFINE_MAX_ASPECT)):
            filtered_offline += 1
            continue
        # Parse frame from filename
        import re
        m = re.search(r"frame_(\d+)", img_path.stem)
        frame_no = int(m.group(1)) if m else -1
        emb = extract_embedding(img, sess, iname, config.REID_SIZE)
        up, lo = extract_color(img)
        track.add_crop(img_path, emb, up, lo, frame_no)
    if track.size() > 0:
        track.build_prototype()
        tracks.append(track)
    else:
        print(f"  Track {lid}: all crops filtered out.")

print(f"\nTotal tracks retained: {len(tracks)}")
print(f"Crops filtered offline: {filtered_offline}")
for tr in tracks[:5]:
    print(f"  Track L{tr.track_id}: {tr.size()} images, coherence {tr.coherence:.3f}")

Loading tracks:   0%|          | 0/24 [00:00<?, ?it/s]

  Track 14: all crops filtered out.
  Track 16: all crops filtered out.
  Track 20: all crops filtered out.
  Track 22: all crops filtered out.
  Track 34: all crops filtered out.

Total tracks retained: 19
Crops filtered offline: 133
  Track L1: 289 images, coherence 0.901
  Track L2: 154 images, coherence 0.929
  Track L3: 273 images, coherence 0.937
  Track L4: 74 images, coherence 0.916
  Track L5: 203 images, coherence 0.903


In [7]:
# ---- Greedy merge of track clusters ----
# Sort by size descending to start with largest
tracks.sort(key=lambda t: t.size(), reverse=True)
final_clusters = []

def merge_clusters(a, b):
    merged = TrackProto(-1)  # placeholder id
    merged.image_paths = a.image_paths + b.image_paths
    merged.embeddings = a.embeddings + b.embeddings
    merged.upper_hist = a.upper_hist + b.upper_hist
    merged.lower_hist = a.lower_hist + b.lower_hist
    merged.frames = a.frames + b.frames
    merged.build_prototype()
    return merged

print("Starting cross-track merging...")
for track in tqdm(tracks, desc="Merging"):
    if track.coherence is not None and track.coherence < COHERENCE_MIN:
        print(f"  Track L{track.track_id} coherence {track.coherence:.3f} < {COHERENCE_MIN} → kept separate")
        final_clusters.append(track)
        continue

    best_idx = -1
    best_score = -1.0
    for i, fc in enumerate(final_clusters):
        overlap = len(track.frame_set().intersection(fc.frame_set()))
        if overlap > MAX_SAME_FRAME_OVERLAP:
            continue
        reid, col, score = track.similarity_to(fc)
        # Apply large folder bonus
        req_reid = MERGE_REID_THRESHOLD
        req_col = MERGE_COLOR_THRESHOLD
        if track.size() >= LARGE_FOLDER_SIZE or fc.size() >= LARGE_FOLDER_SIZE:
            req_reid = LARGE_MERGE_REID_THRESHOLD
            req_col = LARGE_MERGE_COLOR_THRESHOLD
        if reid >= req_reid and col >= req_col:
            if score > best_score:
                best_score = score
                best_idx = i
    if best_idx >= 0:
        final_clusters[best_idx] = merge_clusters(final_clusters[best_idx], track)
        print(f"  Track L{track.track_id} → merged into cluster (size {final_clusters[best_idx].size()}, score {best_score:.3f})")
    else:
        final_clusters.append(track)
        print(f"  Track L{track.track_id} → new cluster (size {track.size()})")

print(f"\nFinal clusters: {len(final_clusters)}")

Starting cross-track merging...


Merging:   0%|          | 0/19 [00:00<?, ?it/s]

  Track L1 → new cluster (size 289)
  Track L3 → new cluster (size 273)
  Track L5 → new cluster (size 203)
  Track L2 → new cluster (size 154)
  Track L27 → new cluster (size 140)
  Track L9 → new cluster (size 89)
  Track L4 → new cluster (size 74)
  Track L6 → new cluster (size 52)
  Track L18 → new cluster (size 50)
  Track L29 → new cluster (size 38)
  Track L32 → new cluster (size 33)
  Track L10 → new cluster (size 21)
  Track L25 → new cluster (size 15)
  Track L26 → new cluster (size 9)
  Track L13 → new cluster (size 6)
  Track L15 → new cluster (size 4)
  Track L24 → new cluster (size 2)
  Track L39 → new cluster (size 2)
  Track L21 → new cluster (size 1)

Final clusters: 19


In [8]:
import shutil

# ---- Write refined person folders ----
refined_dir = out_base / "refined"
refined_dir.mkdir(parents=True, exist_ok=True)

final_clusters.sort(key=lambda c: c.size(), reverse=True)
out_idx = 0
for cluster in final_clusters:
    if cluster.size() < MIN_FOLDER_IMAGES_FINAL:
        print(f"Dropping cluster of size {cluster.size()} (< {MIN_FOLDER_IMAGES_FINAL})")
        continue
    out_idx += 1
    person_dir = refined_dir / f"person_{out_idx:03d}"
    person_dir.mkdir(exist_ok=True)
    for src in cluster.image_paths:
        dest = person_dir / src.name
        # Avoid name collision
        if dest.exists():
            i = 1
            while (person_dir / f"{src.stem}_{i}{src.suffix}").exists():
                i += 1
            dest = person_dir / f"{src.stem}_{i}{src.suffix}"
        shutil.copy2(src, dest)

print(f"Refined identities written: {out_idx}")

Dropping cluster of size 15 (< 18)
Dropping cluster of size 9 (< 18)
Dropping cluster of size 6 (< 18)
Dropping cluster of size 4 (< 18)
Dropping cluster of size 2 (< 18)
Dropping cluster of size 2 (< 18)
Dropping cluster of size 1 (< 18)
Refined identities written: 12


## Final Visualization

In [9]:
import shutil
from matplotlib import pyplot as plt  # only if needed, but we'll skip display

# Build mapping (frame, local_id) -> person_id
identity_map = {}
for folder in sorted(refined_dir.iterdir()):
    if not folder.is_dir() or not folder.name.startswith("person_"):
        continue
    pid = int(folder.name.split("_")[1])
    for img_file in folder.iterdir():
        if img_file.suffix.lower() not in {".jpg",".jpeg",".png"}:
            continue
        m = re.search(r"frame_(\d+).*_l(\d+)", img_file.stem)
        if m:
            frame_no = int(m.group(1))
            local_id = int(m.group(2))
            identity_map[(frame_no, local_id)] = pid

print(f"Loaded {len(set(identity_map.values()))} refined persons for visualization.")

# Visualize video
cap = cv2.VideoCapture(str(VIDEO_PATH))
out_video_path = out_base / "final_visualization.mp4"
out_vid = cv2.VideoWriter(str(out_video_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
tracker_vis = YOLOTracker()

def id_color(id_val):
    hue = ((id_val * 0.61803398875) % 1.0) * 179.0
    hsv = np.array([[[hue, 220, 255]]], dtype=np.uint8)
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0,0]
    return int(bgr[0]), int(bgr[1]), int(bgr[2])

frame_idx = 0
pbar = tqdm(total=total_frames, desc="Rendering")
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1
    dets = tracker_vis.track(frame, persist=True)
    annotated = frame.copy()
    for det in dets:
        x1,y1,x2,y2 = det['bbox']
        lid = det['local_id']
        pid = identity_map.get((frame_idx, lid))
        color = id_color(pid) if pid is not None else (128,128,128)
        label = f"P{pid}" if pid is not None else "?"
        cv2.rectangle(annotated, (x1,y1), (x2,y2), color, 2)
        cv2.putText(annotated, label, (x1, max(y1-10, 20)),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, color, 1, cv2.LINE_AA)
    out_vid.write(annotated)
    pbar.update(1)

pbar.close()
cap.release()
out_vid.release()
print(f"Visualization saved to {out_video_path}")

Loaded 12 refined persons for visualization.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
); using CPU instead.


Rendering:   0%|          | 0/396 [00:00<?, ?it/s]

Visualization saved to outputs/full_pipeline_cam1/final_visualization.mp4


In [10]:
print("\nPipeline complete.")


Pipeline complete.
